# 5 · Modelling

| | |
|---|---|
| Lifecycle stage | 5 of 8: *Modelling* (feature engineering, algorithm comparison, tuning, cross-validation) |
| Tools | scikit-learn, XGBoost, LightGBM, CatBoost (comparison only), MLflow |
| Code | `preprocess.py`, `models.py`, `tuning.py`, `train.py`, `selection.py`, `tracking.py` |
| Outputs | `artifacts/model_*.joblib`, `artifacts/fraudlens_model.joblib` (champion), `reports/model_results.json`, MLflow runs and registered model `fraudlens-champion` |

**Protocol** (fixed before training): tune with *time-series* cross-validation inside the training period,
fit on train, choose the champion and its threshold on validation, and score the test period **once**.

In [1]:
import os, sys, json, warnings
from pathlib import Path
ROOT = Path(os.environ.get("FRAUDLENS_ROOT") or (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())).resolve()
CODE = Path(os.environ.get("FRAUDLENS_CODE", ROOT)).resolve()
sys.path.insert(0, str(CODE / "src"))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

from fraudlens import viz
from fraudlens.utils import load_config, read_json, read_table

viz.setup()
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"   # small notebooks: plotly.js from the CDN
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
cfg = load_config(root=ROOT)
P = cfg["paths"]
print(f"Project root: {ROOT.name} | data present: {(P['silver'] / 'transactions.parquet').exists()}")
from fraudlens import models as M
from fraudlens import tuning
from fraudlens.train import load_view
res = read_json(P["reports"] / "model_results.json")
print(res["protocol"])

Project root: fraudlens | data present: True


tuned with time-series CV inside train; fitted on train; champion and threshold chosen on validation; test scored once


## 5.1 Feature engineering for two model families

`preprocess.py` builds two views of the same gold features, both fitted on **train only**:

* **tree view** (gradient boosting): raw numeric features plus native categorical codes; no scaling needed.
* **linear view** (logistic regression, Isolation Forest): log transforms, train-fitted quantile clipping,
  standardisation, cyclical encoding of hour/day (sin/cos), one-hot categories, missing-value indicators.

In [2]:
pre = read_json(P["reports"] / "preprocessing_report.json")
print("fit on:", pre["fit_on"], "| class counts (train):", pre["class_counts_train"],
      "| suggested scale_pos_weight:", pre["suggested_scale_pos_weight"])
print("excluded features:", pre["excluded_features"], "| added:", pre["added_features"])
for name, v in pre["views"].items():
    print(f"\n{name} view: {v['n_features']} features")
    print(", ".join(v["feature_names"]))

fit on: train | class counts (train): {'legit': 1178257, 'fraud': 6837} | suggested scale_pos_weight: 172.3
excluded features: ['card_prior_txn_count', 'card_prior_category_count', 'distance_km', 'is_new_merchant_for_card'] | added: ['category_share_card']

linear view: 39 features
secs_since_last_txn, amt_to_card_mean, txn_count_1h, txn_count_24h, txn_count_7d, amt_sum_1h, amt_sum_24h, amt_sum_7d, missingindicator_secs_since_last_txn, missingindicator_amt_to_card_mean, amt_zscore_card, missingindicator_amt_zscore_card, category_share_card, missingindicator_category_share_card, log_amount, age, log_city_pop, hour_sin, hour_cos, day_of_week_sin, day_of_week_cos, is_weekend, is_night, category_entertainment, category_food_dining, category_gas_transport, category_grocery_net, category_grocery_pos, category_health_fitness, category_home, category_kids_pets, category_misc_net, category_misc_pos, category_personal_care, category_shopping_net, category_shopping_pos, category_travel, gender_F,

## 5.2 Candidate algorithms and class imbalance

Imbalance is handled with **class weights** (no SMOTE or other resampling): weighting keeps the real
data distribution, so scores stay meaningful as fraud rates, and it is cheap on 1.2 M rows.

| Model | Why it is a candidate |
|---|---|
| Logistic regression | Transparent baseline; tells us how much non-linearity buys |
| HistGradientBoosting (scikit-learn) | Fast gradient boosting with native categorical support, no extra dependency |
| XGBoost, LightGBM | State-of-the-art boosted trees for tabular data |
| Isolation Forest | Unsupervised anomaly detector trained on legitimate transactions only; finds "unusual", not "fraud-like" |
| Hybrid | HistGradientBoosting with the Isolation Forest anomaly score as an extra feature |

In [3]:
cands = pd.DataFrame({"library": {m: r["library"] for m, r in res["results"].items()},
                      "view": {m: r["view"] for m, r in res["results"].items()},
                      "imbalance handling": res["imbalance_handling"],
                      "train seconds": res["train_seconds"]})
cands

,library,view,imbalance handling,train seconds
logistic,scikit-learn,linear,class_weight='balanced'; no resampling,6.0000
hist_gb,scikit-learn,tree,class_weight='balanced'; no resampling,393.1000
xgboost,xgboost,tree,scale_pos_weight=172.3 (class weighting); no r...,453.5000
lightgbm,lightgbm,tree,class_weight='balanced'; no resampling,"22,846.7000"
isolation_forest,scikit-learn,linear,"none needed: unsupervised, fitted on legitimat...",23.2000
hybrid_gb,scikit-learn,tree,class_weight='balanced'; no resampling,43.6000


## 5.3 Hyper-parameter tuning with time-series cross-validation

Random K-fold would train on the future. `tuning.time_folds` cuts the (time-ordered) training period into
blocks and validates each block on a model trained on everything **before** it (expanding window).

In [4]:
folds = tuning.time_folds(1_185_094, 3)
fig, ax = plt.subplots(figsize=(9, 1.9))
for i, (tr, va) in enumerate(folds):
    ax.barh(i, len(tr), color=viz.SERIES[0]); ax.barh(i, len(va), left=len(tr), color=viz.SERIES[1])
ax.set_yticks(range(3), [f"fold {i + 1}" for i in range(3)]); ax.invert_yaxis()
ax.set_xlabel("training rows in time order"); ax.set_title("Expanding-window CV: blue = fit, orange = validate")
plt.show()

rows = []
for m, t in res["tuning"].items():
    for c in t["candidates"]:
        rows.append({"model": m, "candidate": c["candidate"], **c["params"],
                     "cv_pr_auc_mean": c.get("cv_pr_auc_mean"), "cv_pr_auc_std": c.get("cv_pr_auc_std")})
tun = pd.DataFrame(rows)
print({m: {"best": t["best_params"], "cv_pr_auc": round(t["best_cv_pr_auc_mean"], 4), "±": round(t["best_cv_pr_auc_std"], 4)}
       for m, t in res["tuning"].items()})
tun.sort_values(["model", "cv_pr_auc_mean"], ascending=[True, False]).groupby("model").head(4)

{'hist_gb': {'best': {'learning_rate': 0.1, 'max_leaf_nodes': 15, 'min_samples_leaf': 50, 'l2_regularization': 1.0}, 'cv_pr_auc': 0.9724, '±': 0.0106}, 'xgboost': {'best': {'n_estimators': 800, 'learning_rate': 0.06, 'max_depth': 8, 'min_child_weight': 1, 'subsample': 0.85, 'colsample_bytree': 0.85, 'reg_lambda': 3.0}, 'cv_pr_auc': 0.9703, '±': 0.0145}, 'lightgbm': {'best': {'n_estimators': 400, 'learning_rate': 0.06, 'num_leaves': 31, 'min_child_samples': 50, 'subsample': 0.85, 'colsample_bytree': 0.85, 'reg_lambda': 3.0}, 'cv_pr_auc': 0.9745, '±': 0.0112}}


,model,candidate,learning_rate,max_leaf_nodes,min_samples_leaf,l2_regularization,cv_pr_auc_mean,cv_pr_auc_std,n_estimators,max_depth,min_child_weight,subsample,colsample_bytree,reg_lambda,num_leaves,min_child_samples
8,hist_gb,8,0.1000,15.0000,50.0000,1.0000,0.9724,0.0106,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,hist_gb,2,0.1000,15.0000,20.0000,3.0000,0.9718,0.0086,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,hist_gb,0,0.0600,31.0000,50.0000,1.0000,0.9706,0.0109,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,hist_gb,3,0.1000,31.0000,50.0000,1.0000,0.9671,0.0169,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,lightgbm,0,0.0600,NaN,NaN,NaN,0.9745,0.0112,400.0000,NaN,NaN,0.8500,0.8500,3.0000,31.0000,50.0000
20,lightgbm,2,0.0300,NaN,NaN,NaN,0.9742,0.0100,800.0000,NaN,NaN,0.8500,0.8500,3.0000,15.0000,100.0000
22,lightgbm,4,0.1000,NaN,NaN,NaN,0.9739,0.0126,400.0000,NaN,NaN,1.0000,1.0000,1.0000,63.0000,50.0000
26,lightgbm,8,0.0300,NaN,NaN,NaN,0.9734,0.0125,800.0000,NaN,NaN,0.8500,0.8500,3.0000,63.0000,100.0000
12,xgboost,3,0.0600,NaN,NaN,NaN,0.9703,0.0145,800.0000,8.0000,1.0000,0.8500,0.8500,3.0000,NaN,NaN
9,xgboost,0,0.0600,NaN,NaN,NaN,0.9698,0.0126,400.0000,6.0000,5.0000,0.8500,0.8500,3.0000,NaN,NaN


## 5.4 Live algorithm comparison (time-series CV)

A compact re-run of the comparison so it can be reproduced in minutes: every 4th training transaction
(time order preserved), three expanding-window folds, default settings, and **CatBoost** added as an
extra contender. The full-size, tuned comparison from the pipeline follows in 5.5.

In [5]:
import time
from sklearn.metrics import average_precision_score

tree, lin = load_view(cfg, "tree")["train"], load_view(cfg, "linear")["train"]
step = 4
Xt, yt = tree["X"].iloc[::step].reset_index(drop=True), tree["y"][::step]
Xl = lin["X"].iloc[::step].reset_index(drop=True)
cat_cols = [c for c in M.CATEGORICAL_IN_TREE_VIEW if c in Xt.columns]
print(f"{len(Xt):,} rows, {int(yt.sum()):,} frauds")

def make(name):
    if name == "catboost":
        from catboost import CatBoostClassifier
        return CatBoostClassifier(iterations=300, learning_rate=0.1, depth=6, auto_class_weights="Balanced",
                                  cat_features=cat_cols, verbose=0, random_seed=42, thread_count=4,
                                  allow_writing_files=False)
    m = M.build(name, 42, categorical=cat_cols, scale_pos_weight=pre["suggested_scale_pos_weight"])
    if name == "lightgbm":
        m.set_params(n_jobs=2)   # avoids OpenMP thread contention with XGBoost in the same process
    return m

import importlib.util
names = ["logistic", "hist_gb", "xgboost", "lightgbm"] + (["catboost"] if importlib.util.find_spec("catboost") else [])
rows = []
for name in names:
    X = Xl if M.VIEW_BY_MODEL.get(name) == "linear" else Xt
    for k, (tr, va) in enumerate(tuning.time_folds(len(X), 3)):
        t0 = time.perf_counter()
        model = make(name)
        M.fit(model, name, X.iloc[tr], yt[tr], categorical=cat_cols if name == "lightgbm" else None)
        ap = average_precision_score(yt[va], M.positive_scores(model, X.iloc[va]))
        rows.append({"model": name, "fold": k + 1, "pr_auc": ap, "seconds": time.perf_counter() - t0})
cv = pd.DataFrame(rows)
cv_summary = cv.groupby("model").agg(pr_auc_mean=("pr_auc", "mean"), pr_auc_std=("pr_auc", "std"),
                                     seconds=("seconds", "sum")).sort_values("pr_auc_mean", ascending=False)
cv_summary

296,274 rows, 1,704 frauds


,pr_auc_mean,pr_auc_std,seconds
model,,,
lightgbm,0.9630,0.0182,34.1129
xgboost,0.9590,0.0174,25.6490
catboost,0.9537,0.0190,117.5704
hist_gb,0.9274,0.0415,29.3780
logistic,0.6531,0.0320,2.7491


In [6]:
fig, ax = plt.subplots(figsize=(8, 3.4))
order = cv_summary.index.tolist()
for i, m in enumerate(order):
    v = cv.loc[cv["model"] == m, "pr_auc"]
    ax.scatter(v, [i] * len(v), color=viz.model_color(m) if m != "catboost" else viz.SERIES[6], s=40)
    ax.plot([v.mean()] * 2, [i - 0.3, i + 0.3], color=viz.INK)
ax.set_yticks(range(len(order)), order); ax.invert_yaxis()
ax.set_xlabel("PR-AUC per fold (line = mean)"); ax.set_title("Time-series CV on a 1-in-4 sample of the training period")
plt.show()

## 5.5 Full comparison from the pipeline (tuned, full training data)

Validation PR-AUC decides; test is shown for completeness but was not used for any choice.

In [7]:
comp = pd.DataFrame(res["comparison"]).set_index("model")
display(comp)
fig, ax = plt.subplots(figsize=(8, 3.6))
c2 = comp.sort_values("val_pr_auc")
ax.barh(np.arange(len(c2)) + 0.2, c2["val_pr_auc"], height=0.4, color=viz.SERIES[0], label="validation")
ax.barh(np.arange(len(c2)) - 0.2, c2["test_pr_auc"], height=0.4, color=viz.SERIES[1], label="test")
ax.set_yticks(range(len(c2)), c2.index); ax.set_xlabel("PR-AUC"); ax.legend(loc="lower right")
ax.set_title("Supervised boosted trees dominate; the anomaly detector alone is far behind")
plt.show()

,library,cv_pr_auc,val_pr_auc,test_pr_auc,test_roc_auc,test_precision,test_recall,test_alerts_per_day,test_false_alarms_per_catch,test_net_benefit
model,,,,,,,,,,
lightgbm,lightgbm,0.9745,0.9878,0.9775,0.9995,0.8934,0.9529,11.8000,0.1000,"967,973.0900"
xgboost,xgboost,0.9703,0.9868,0.9768,0.9993,0.8873,0.9506,11.9000,0.1000,"963,139.0700"
hybrid_gb,scikit-learn,NaN,0.9841,0.9753,0.9996,0.8989,0.9487,11.7000,0.1000,"966,349.1300"
hist_gb,scikit-learn,0.9724,0.9820,0.9698,0.9995,0.7240,0.9622,14.7000,0.4000,"964,181.8600"
logistic,scikit-learn,NaN,0.6839,0.5900,0.9822,0.2029,0.8308,45.4000,3.9000,"822,771.7700"
isolation_forest,scikit-learn,NaN,0.3556,0.2662,0.9055,0.0711,0.5716,89.1000,13.1000,"518,303.5400"


## 5.6 Champion selection: the simplest non-inferior model

Rule (registered in advance in `selection.py`): rank models by complexity
(`logistic < hist_gb < lightgbm < xgboost < hybrid`) and choose the **simplest** one whose validation
PR-AUC is not worse than the best by more than **0.01**, judged by the upper bound of a 95% paired
bootstrap confidence interval of the gap. This avoids paying operational complexity (extra libraries,
a second model in the hybrid) for differences that are within noise.

In [8]:
sel = res["selection"]
print(sel["reason"])
gaps = pd.DataFrame(sel["comparisons"]).T.loc[[m for m in sel["complexity_order"] if m in sel["comparisons"]]]
display(gaps)
fig, ax = plt.subplots(figsize=(7, 2.8))
for i, (m, r) in enumerate(gaps.iterrows()):
    ax.errorbar(r["gap_mean"], i, xerr=[[r["gap_mean"] - r["ci95_low"]], [r["ci95_high"] - r["gap_mean"]]],
                fmt="o", color=viz.SERIES[1] if m == res["champion"] else viz.MUTED, capsize=4)
ax.axvline(sel["margin"], ls="--", color=viz.INK_2); ax.annotate(" margin 0.01", (sel["margin"], len(gaps) - 0.5))
ax.set_yticks(range(len(gaps)), gaps.index); ax.invert_yaxis(); ax.set_xlim(-0.005, 0.04)
ax.set_xlabel("PR-AUC gap to the best model (95% bootstrap CI)"); ax.set_title(f"Champion: {res['champion']}")
plt.show()

lightgbm has the highest validation PR-AUC (0.9878), but hist_gb (0.9820) is non-inferior: the 95% bootstrap interval of the gap is [+0.0032, +0.0084], entirely below the 0.01 margin. hist_gb is simpler, so it is selected.


,gap_mean,ci95_low,ci95_high,equivalent_to_best,n_boot
logistic,0.3037,0.2734,0.3357,False,500
hist_gb,0.0059,0.0032,0.0084,True,500
lightgbm,0.0000,0.0000,0.0000,True,0
xgboost,0.0011,-0.0013,0.0042,True,500
hybrid_gb,0.0038,0.0020,0.0058,True,500


## 5.7 The deployable champion

The champion, its fitted preprocessor and the validation-chosen threshold are saved together as one
`FraudModel` object, so training and serving cannot drift apart. The pipeline also proves that the saved
object scores the stored training matrices **exactly** (train/serve parity test).

In [9]:
from fraudlens.scoring import FraudModel
fm = FraudModel.load(P["artifacts"] / "fraudlens_model.joblib")
print(f"champion={fm.champion}  view={fm.view}  threshold={fm.threshold:.4f} ({fm.threshold_strategy})")
print(fm.model)

champion=hist_gb  view=tree  threshold=0.7397 (min_cost)
HistGradientBoostingClassifier(categorical_features=['category', 'gender'],
                               class_weight='balanced', early_stopping=True,
                               l2_regularization=1.0, max_iter=400,
                               max_leaf_nodes=15, min_samples_leaf=50,
                               n_iter_no_change=20, random_state=42)


## 5.8 Experiment tracking: MLflow

Every training run is logged to MLflow (parameters, nested tuning runs, metrics, figures, the model), and
the champion is registered as `fraudlens-champion` with the alias `champion`.

In [10]:
import mlflow
from mlflow.tracking import MlflowClient
info = res.get("mlflow", {})
uri = f"sqlite:///{(ROOT / 'mlruns' / 'mlflow.db').as_posix()}"
if (ROOT / "mlruns" / "mlflow.db").exists():
    mlflow.set_tracking_uri(uri)
    runs = mlflow.search_runs(experiment_names=[cfg["platform"]["mlflow_experiment"]], max_results=200)
    cols = [c for c in ["run_id", "tags.mlflow.runName", "status", "metrics.val_pr_auc", "metrics.test_pr_auc"] if c in runs]
    print(f"{len(runs)} runs in experiment '{cfg['platform']['mlflow_experiment']}'")
    display(runs[cols].head(12))
    client = MlflowClient()
    for v in client.search_model_versions(f"name='{cfg['platform']['registered_model']}'"):
        print(f"registered: {v.name} v{v.version} (run {v.run_id[:8]}) aliases={getattr(v, 'aliases', [])}")
else:
    print("No local MLflow store here; tracking info from the run:", info)

30 runs in experiment 'fraudlens'


,run_id,tags.mlflow.runName,status
0,cb52a8dc07954dcbaa89bea2d52a71fc,monitor-20261018-094826,FINISHED
1,bea48a175e754018af428af6f00988e4,monitor-20261018-091751,FINISHED
2,d145cd00ae1746ff86793f2eee21e37f,tune-lightgbm-8,FINISHED
3,6f109afdfc64445dbf2dd6a32a332f27,tune-lightgbm-7,FINISHED
4,cd8590fb80d94189af04b523d9072689,tune-lightgbm-6,FINISHED
5,5812e2a66e254c9598d982f32e51bf1f,tune-lightgbm-5,FINISHED
6,9caa462ce9fb449ba496516389e15fb7,tune-lightgbm-4,FINISHED
7,16096f7f6c1847d99e111914a6a224ed,tune-lightgbm-3,FINISHED
8,116c96c291fd400a8a965a8631e5d07c,tune-lightgbm-2,FINISHED
9,8c9e4a1c8dc544b38d784c6b20190a26,tune-lightgbm-1,FINISHED


registered: fraudlens-champion v1 (run 348c68b9) aliases=[]


## 5.9 Summary

* Boosted trees reach validation PR-AUC ≈ 0.98 (lift ≈ 164× over random); logistic regression ≈ 0.68 and
  the unsupervised Isolation Forest ≈ 0.36, so the patterns are strongly non-linear but well captured by trees.
* LightGBM, XGBoost, HistGradientBoosting and the hybrid are statistically within the 0.01 margin; the rule
  picks **HistGradientBoosting** (scikit-learn only, smallest artefact, fastest to serve).
* The anomaly score adds nothing measurable on top of the supervised model on this dataset; it remains
  valuable as a detector of *new* patterns and is monitored, but it is not in the deployed path.

**Next:** [06 · Evaluation & interpretation](06_evaluation_interpretation.ipynb)